# 10-K NLP Feature Construction
Three parallel tracks → merged panel → cross-sectional analysis

| Track | Source section | Method | Output |
|---|---|---|---|
| 1 | item_1A, item_7 | Loughran-McDonald sentiment | tone scores per (cik, year) |
| 2 | item_1 + item_1A + item_7 | TF-IDF cosine similarity YoY | text-change score per (cik, year) |
| 3 | item_1, item_7 | Strategy-category regex counts | shift scores per (cik, year) |

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

## 1. Configuration

In [ ]:
import os

CONFIG = {
    # Folder on Drive where {cik}/{cik}_filings.parquet files live
    'filings_folder': '/content/drive/MyDrive/SEC_Filings',

    # Loughran-McDonald master dictionary (uploaded to Drive or local)
    'lm_dict_path': '/content/drive/MyDrive/SEC_Filings/Loughran-McDonald_MasterDictionary_1993-2024.csv',

    # Strategy-category dictionary with regex patterns
    'strategy_dict_path': '/content/drive/MyDrive/SEC_Filings/dictionary.csv',

    # Lazy Prices return dataset (cik, accession, fdate, permno, monthly_return, ...)
    'lazy_prices_path': '/content/drive/MyDrive/SEC_Filings/lazy_prices_dataset.csv',

    # Where to save outputs
    'output_folder': '/content/drive/MyDrive/SEC_Filings',

    # Sections to include in cosine similarity (Track 2)
    'sim_sections': ['item_1', 'item_1A', 'item_7'],

    # Year range to process
    'start_year': 2004,
    'end_year':   2025,
}
print('Config OK')

## 2. Dependencies

In [ ]:
# Run once per Colab session
import subprocess
subprocess.run(['pip', 'install', '-q', 'pyarrow', 'fastparquet', 'scikit-learn', 'tqdm'], check=True)
print('Dependencies ready')

## 3. Load all filing parquets from Drive

In [ ]:
import glob
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

filings_folder = CONFIG['filings_folder']

pq_files = sorted(glob.glob(os.path.join(filings_folder, '**', '*_filings.parquet'), recursive=True))
print(f'Found {len(pq_files)} parquet files')

dfs = []
for p in tqdm(pq_files, desc='Loading parquets'):
    try:
        df = pd.read_parquet(p)
        dfs.append(df)
    except Exception as e:
        print(f'  Skip {p}: {e}')

filings = pd.concat(dfs, ignore_index=True)
filings['year'] = filings['year'].astype('Int64')

# Keep only rows within year range and with at least one text section
yr = CONFIG['start_year'], CONFIG['end_year']
filings = filings[filings['year'].between(yr[0], yr[1])]

text_cols = ['item_1', 'item_1A', 'item_7']
for c in text_cols:
    if c not in filings.columns:
        filings[c] = ''
    filings[c] = filings[c].fillna('').astype(str)

# Deduplicate: keep latest filing per (cik, year)
filings = (
    filings.sort_values('filing_date', na_position='first')
           .drop_duplicates(subset=['cik', 'year'], keep='last')
           .reset_index(drop=True)
)

print(f'\nLoaded {len(filings):,} filings across {filings["cik"].nunique():,} companies')
print(f'Year range: {filings["year"].min()} – {filings["year"].max()}')
filings[['cik', 'company', 'year', 'filing_date']].head(8)

## 4. Track 1 — Loughran-McDonald Sentiment Scoring

Counts tone words (negative, positive, uncertainty, litigious, strong/weak modal, constraining)
in **item_1A** (risk factors) and **item_7** (MD&A) separately, normalised by total word count.

In [ ]:
import re

# ── Load LM dictionary into category → frozenset(words) ──
lm_raw = pd.read_csv(CONFIG['lm_dict_path'])
lm_raw['Word'] = lm_raw['Word'].str.upper()

LM_CATEGORIES = ['Negative', 'Positive', 'Uncertainty', 'Litigious',
                 'Strong_Modal', 'Weak_Modal', 'Constraining']

lm_sets: dict[str, frozenset] = {
    cat: frozenset(lm_raw.loc[lm_raw[cat] != 0, 'Word'].tolist())
    for cat in LM_CATEGORIES
}

total_lm_words = sum(len(s) for s in lm_sets.values())
print(f'LM dictionary loaded: {len(lm_raw):,} words')
for cat, s in lm_sets.items():
    print(f'  {cat}: {len(s):,}')

In [ ]:
_TOKENISE = re.compile(r'[A-Z]+')

def lm_scores(text: str) -> dict[str, float]:
    """Return normalised LM category scores for a single text string."""
    tokens = _TOKENISE.findall(text.upper())
    n = len(tokens)
    if n == 0:
        return {cat: 0.0 for cat in LM_CATEGORIES}
    counts = {cat: 0 for cat in LM_CATEGORIES}
    for tok in tokens:
        for cat, words in lm_sets.items():
            if tok in words:
                counts[cat] += 1
    return {cat: counts[cat] / n for cat in LM_CATEGORIES}


def score_section(df: pd.DataFrame, col: str, prefix: str) -> pd.DataFrame:
    """Apply lm_scores to column `col`, return DataFrame with prefixed columns."""
    records = []
    for text in tqdm(df[col], desc=f'LM scoring {prefix}', leave=False):
        records.append(lm_scores(text))
    scores = pd.DataFrame(records, index=df.index)
    scores.columns = [f'{prefix}_{c.lower()}' for c in scores.columns]
    return scores


# Score item_1A (risk) and item_7 (MD&A)
lm_1a   = score_section(filings, 'item_1A', 'risk')
lm_item7 = score_section(filings, 'item_7',  'mda')

sentiment_df = pd.concat(
    [filings[['cik', 'year']].reset_index(drop=True),
     lm_1a.reset_index(drop=True),
     lm_item7.reset_index(drop=True)],
    axis=1
)
print(f'\nSentiment features: {sentiment_df.shape}')
sentiment_df.head(4)

## 5. Track 2 — Text Change Score (Lazy Prices / Cosine Similarity)

For each company, computes the TF-IDF cosine similarity between the combined 10-K text of year *t*
and year *t-1*. A score near 1 means the filing is nearly identical ("lazy"); near 0 means a large
rewrite. The **text change score** is `1 - similarity`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Concatenate sections for each filing
filings['_full_text'] = (
    filings['item_1'].str[:8000] + ' ' +
    filings['item_1A'].str[:8000] + ' ' +
    filings['item_7'].str[:8000]
)

similarity_records = []

for cik, grp in tqdm(filings.groupby('cik'), desc='Cosine similarity', leave=True):
    grp = grp.sort_values('year').reset_index(drop=True)
    if len(grp) < 2:
        continue

    texts = grp['_full_text'].tolist()
    years = grp['year'].tolist()

    # Fit TF-IDF on all filings for this company
    try:
        tfidf = TfidfVectorizer(
            max_features=5000,
            sublinear_tf=True,
            stop_words='english',
            min_df=1,
        )
        mat = tfidf.fit_transform(texts)
    except ValueError:
        continue

    for i in range(1, len(grp)):
        sim = float(cosine_similarity(mat[i], mat[i - 1])[0, 0])
        similarity_records.append({
            'cik':        cik,
            'year':       years[i],
            'text_sim':   round(sim, 6),
            'text_change': round(1.0 - sim, 6),
        })

similarity_df = pd.DataFrame(similarity_records)
print(f'\nSimilarity features: {similarity_df.shape}')
print(similarity_df['text_change'].describe().round(4).to_string())
similarity_df.head(4)

## 6. Track 3 — Strategy Shift (Regex Dictionary)

Counts mentions of each strategy category (from `dictionary.csv`) in **item_1** and **item_7**,
normalised by word count. Year-over-year change in each category is the strategy shift signal.

In [ ]:
import warnings

strat_dict = pd.read_csv(CONFIG['strategy_dict_path'])
print(f'Strategy dictionary: {len(strat_dict)} terms across {strat_dict["Category"].nunique()} categories')
print(strat_dict['Category'].value_counts().to_string())

In [ ]:
# Compile one regex per category (OR of all patterns in that category)
category_patterns: dict[str, re.Pattern] = {}
for cat, grp in strat_dict.groupby('Category'):
    patterns = grp['Suggested_Regex'].dropna().tolist()
    combined = '|'.join(f'(?:{p})' for p in patterns if p)
    if combined:
        try:
            category_patterns[cat] = re.compile(combined, re.IGNORECASE)
        except re.error as e:
            print(f'Bad regex for category {cat}: {e}')

print(f'Compiled {len(category_patterns)} category patterns')


def strategy_scores(text: str) -> dict[str, float]:
    """Return mention-rate per strategy category, normalised by word count."""
    words = len(text.split())
    if words == 0:
        return {cat: 0.0 for cat in category_patterns}
    return {
        cat: len(pat.findall(text)) / words
        for cat, pat in category_patterns.items()
    }


def score_strategy_section(df: pd.DataFrame, col: str, prefix: str) -> pd.DataFrame:
    records = []
    for text in tqdm(df[col], desc=f'Strategy scoring {prefix}', leave=False):
        records.append(strategy_scores(text))
    scores = pd.DataFrame(records, index=df.index)
    cat_cols = {c: f'{prefix}_{c.lower().replace(" ", "_").replace("&", "and")}' for c in scores.columns}
    scores.rename(columns=cat_cols, inplace=True)
    return scores


strat_item1  = score_strategy_section(filings, 'item_1',  'biz')
strat_item7  = score_strategy_section(filings, 'item_7',  'mda')

strategy_raw = pd.concat(
    [filings[['cik', 'year']].reset_index(drop=True),
     strat_item1.reset_index(drop=True),
     strat_item7.reset_index(drop=True)],
    axis=1
)

# Compute year-over-year change per (cik, category)
strategy_raw = strategy_raw.sort_values(['cik', 'year'])
score_cols = [c for c in strategy_raw.columns if c not in ('cik', 'year')]
strategy_df = strategy_raw.copy()
strategy_df[score_cols] = (
    strategy_raw.groupby('cik')[score_cols]
                .diff()          # t minus t-1
)
strategy_df = strategy_df.dropna(subset=score_cols, how='all')

print(f'\nStrategy-shift features: {strategy_df.shape}')
strategy_df.head(4)

## 7. Merge all NLP features

In [ ]:
nlp_features = (
    sentiment_df
    .merge(similarity_df, on=['cik', 'year'], how='outer')
    .merge(strategy_df,   on=['cik', 'year'], how='outer')
)

nlp_features['cik']  = nlp_features['cik'].astype(str).str.strip()
nlp_features['year'] = nlp_features['year'].astype('Int64')

out_path = os.path.join(CONFIG['output_folder'], 'nlp_features.parquet')
nlp_features.to_parquet(out_path, index=False)

print(f'NLP features: {nlp_features.shape}')
print(f'Saved → {out_path}')
nlp_features.head(4)

## 8. Join with return data (lazy_prices_dataset)

Merge NLP features with the panel of monthly returns and accounting variables.
The filing date is matched to the return month that follows the filing.

In [ ]:
returns = pd.read_csv(CONFIG['lazy_prices_path'])
returns['fdate']        = pd.to_datetime(returns['fdate'])
returns['return_month'] = pd.to_datetime(returns['return_month'])
returns['cik']          = returns['cik'].astype(str).str.strip().str.lstrip('0')

# Filing year = calendar year of the fiscal year end, approximated from fdate
returns['year'] = returns['fdate'].dt.year.astype('Int64')

# Keep one row per (cik, year) for the merge: use the first filing observation
ret_key = (
    returns.sort_values('fdate')
           .drop_duplicates(subset=['cik', 'year'], keep='first')
    [['cik', 'year', 'accession', 'fdate', 'gvkey', 'permno',
      'datadate', 'at', 'sale', 'ni']]
)

# Monthly return panel (keep all return months)
ret_panel = returns[['cik', 'year', 'return_month', 'monthly_return', 'permno']].copy()

# Merge NLP features onto filing-level keys
panel = ret_key.merge(nlp_features, on=['cik', 'year'], how='inner')

# Then re-attach all monthly return rows
panel = ret_panel.merge(
    panel.drop(columns=['permno']),
    on=['cik', 'year'],
    how='inner'
)

panel_path = os.path.join(CONFIG['output_folder'], 'analysis_panel.parquet')
panel.to_parquet(panel_path, index=False)

print(f'Analysis panel: {panel.shape}')
print(f'Unique companies: {panel["cik"].nunique():,}')
print(f'Unique (cik, year): {panel.drop_duplicates(["cik","year"]).shape[0]:,}')
print(f'Saved → {panel_path}')
panel.head(4)

## 9. Portfolio sort preview

Quintile-sort companies each year on a chosen NLP signal and compute average next-month return
per quintile. A monotonic pattern suggests the signal carries return-predictive information.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Pick a signal to inspect ──────────────────────────────────────────────────
SIGNAL = 'text_change'   # ← change to any NLP column to explore

if SIGNAL not in panel.columns:
    print(f'Signal "{SIGNAL}" not found. Available signals:')
    print([c for c in panel.columns if c not in ('cik','year','permno','return_month',
                                                   'monthly_return','accession','fdate',
                                                   'gvkey','datadate','at','sale','ni')])
else:
    # One row per (cik, year): take next-12-month buy-and-hold return
    bahr = (
        panel.groupby(['cik', 'year'])
             .apply(lambda g: (1 + g['monthly_return'].fillna(0)).prod() - 1)
             .reset_index(name='annual_return')
    )

    signal_df = (
        panel.drop_duplicates(['cik', 'year'])[['cik', 'year', SIGNAL]]
             .merge(bahr, on=['cik', 'year'])
             .dropna(subset=[SIGNAL, 'annual_return'])
    )

    # Quintile sort within each year
    signal_df['quintile'] = (
        signal_df.groupby('year')[SIGNAL]
                 .transform(lambda x: pd.qcut(x, 5, labels=False, duplicates='drop') + 1)
    )

    q_returns = (
        signal_df.groupby('quintile')['annual_return']
                 .agg(['mean', 'count'])
                 .reset_index()
    )
    q_returns.columns = ['Quintile', 'Mean Annual Return', 'N']
    q_returns['Mean Annual Return (%)'] = (q_returns['Mean Annual Return'] * 100).round(2)

    # Long-short return (Q5 - Q1)
    ls = q_returns.loc[q_returns['Quintile'] == 5, 'Mean Annual Return (%)'].values[0] -          q_returns.loc[q_returns['Quintile'] == 1, 'Mean Annual Return (%)'].values[0]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(q_returns['Quintile'], q_returns['Mean Annual Return (%)'],
                  color=['#d62728','#ff7f0e','#7f7f7f','#2ca02c','#1f77b4'], alpha=0.85,
                  edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'Quintile sort on: {SIGNAL}
Long–short (Q5−Q1): {ls:+.2f}%', fontsize=13)
    ax.set_xlabel('Quintile (1 = lowest signal, 5 = highest)')
    ax.set_ylabel('Mean Annual Return (%)')
    ax.set_xticks([1, 2, 3, 4, 5])
    for bar, row in zip(bars, q_returns.itertuples()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{row._3:.1f}%
(n={row.N})', ha='center', va='bottom', fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['output_folder'], f'portfolio_sort_{SIGNAL}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print(q_returns[['Quintile', 'Mean Annual Return (%)', 'N']].to_string(index=False))

## 10. Next: Fama-MacBeth regressions

With `analysis_panel.parquet` ready, the next step is a Fama-MacBeth regression:

```python
import linearmodels as lm

# Controls: log(at), log(sale), ni/at, past 12m return (momentum)
# Test whether NLP signals carry incremental return-predictive power
# after controlling for size, book-to-market, and momentum
```

Load `analysis_panel.parquet` in a separate notebook and run cross-sectional regressions
for each signal to obtain t-statistics with Newey-West standard errors.